# Install fastembed and explore the BM25 sparse encoder

In [1]:
from fastembed import SparseTextEmbedding

model = SparseTextEmbedding(model_name="Qdrant/bm25")

sample_texts = [
    "Hypertension treatment guidelines recommend ACE inhibitors as first-line therapy.",
    "Metformin is commonly prescribed for type 2 diabetes management.",
]

sparse_vectors = list(model.embed(sample_texts))

for text, vec in zip(sample_texts, sparse_vectors):
    print(text)
    print(f"  non-zero terms: {len(vec.indices)}")
    print(f"  indices (first 10): {vec.indices[:10]}")
    print(f"  values (first 10): {vec.values[:10]}")
    print()

c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hypertension treatment guidelines recommend ACE inhibitors as first-line therapy.
  non-zero terms: 9
  indices (first 10): [ 168206934 2114294529 2041505284  569095015 1625090372  280431834
 1443829736  640477688 2136367696]
  values (first 10): [1.65209739 1.65209739 1.65209739 1.65209739 1.65209739 1.65209739
 1.65209739 1.65209739 1.65209739]

Metformin is commonly prescribed for type 2 diabetes management.
  non-zero terms: 7
  indices (first 10): [ 148329017  102120688  732425314 2045916435   19522071 1100459356
 2135986242]
  values (first 10): [1.660867 1.660867 1.660867 1.660867 1.660867 1.660867 1.660867]



# Recreate medrag_text with named dense + sparse vectors

In [3]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels

client = QdrantClient(url="http://localhost:6333")
client.get_collections()

TEXT_COLLECTION = "medrag_text"

if client.collection_exists(TEXT_COLLECTION):
    client.delete_collection(TEXT_COLLECTION)
    print(f"Dropped '{TEXT_COLLECTION}'")

client.create_collection(
    collection_name=TEXT_COLLECTION,
    vectors_config={
        "dense": qmodels.VectorParams(size=1536, distance=qmodels.Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": qmodels.SparseVectorParams(
            modifier=qmodels.Modifier.IDF,
        ),
    },
)
print(f"Created '{TEXT_COLLECTION}' with named dense + sparse (IDF-modified) vectors")

client.get_collection(TEXT_COLLECTION)

Dropped 'medrag_text'
Created 'medrag_text' with named dense + sparse (IDF-modified) vectors


CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=0, segments_count=4, config=CollectionConfig(params=CollectionParams(vectors={'dense': VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors={'sparse': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap

# Load everything needed to rebuild points (chunks, dense embeddings, links)

In [4]:
import sys, os
from pathlib import Path
import logging

def find_project_root(marker="backend", start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not find a '{marker}' folder above {current}")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT / "backend"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)

from medrag.embeddings.storage import load_embeddings
from medrag.processing.storage import load_chunks
from medrag.processing.image_linking import load_image_chunk_links
from medrag.embeddings.qdrant_ingestion import dedupe_links, build_link_maps
from run_chunking import WHO_TOPIC_GROUPS

CHUNKS_DIR = str(PROJECT_ROOT / "data" / "processed" / "chunks")
EMBEDDINGS_DIR = str(PROJECT_ROOT / "data" / "processed" / "embeddings")

def load_all_chunks_for_source(source: str, chunks_dir: str) -> dict:
    source_dir = Path(chunks_dir) / source
    lookup = {}
    for filepath in sorted(source_dir.glob("*.jsonl")):
        topic = filepath.stem
        for chunk in load_chunks(source=source, topic=topic, output_dir=chunks_dir):
            lookup.setdefault(chunk.chunk_id, chunk)
    return lookup

def load_who_chunks() -> dict:
    who_topics = [t for group in WHO_TOPIC_GROUPS for t in group]
    lookup = {}
    for topic in who_topics:
        for chunk in load_chunks(source="who", topic=topic, output_dir=CHUNKS_DIR):
            lookup.setdefault(chunk.chunk_id, chunk)
    return lookup

who_chunk_lookup = load_who_chunks()
who_embeddings, who_index = load_embeddings("who", EMBEDDINGS_DIR)

pubmed_chunk_lookup = load_all_chunks_for_source("pubmed", CHUNKS_DIR)
pubmed_embeddings, pubmed_index = load_embeddings("pubmed", EMBEDDINGS_DIR)

openfda_chunk_lookup = load_all_chunks_for_source("openfda", CHUNKS_DIR)
openfda_embeddings, openfda_index = load_embeddings("openfda", EMBEDDINGS_DIR)

links = dedupe_links(load_image_chunk_links(EMBEDDINGS_DIR))
chunk_to_images, _ = build_link_maps(links)

print(f"WHO: {len(who_chunk_lookup)} chunks, {len(who_index)} embeddings")
print(f"PubMed: {len(pubmed_chunk_lookup)} chunks, {len(pubmed_index)} embeddings")
print(f"OpenFDA: {len(openfda_chunk_lookup)} chunks, {len(openfda_index)} embeddings")
print(f"Links: {len(links)} deduped, {len(chunk_to_images)} chunks with images")

Project root: C:\Users\DELL\Desktop\medrag


  Deduped links: 55 -> 50 (5 duplicate mentions removed)


WHO: 4804 chunks, 4804 embeddings
PubMed: 4725 chunks, 4725 embeddings
OpenFDA: 13167 chunks, 13167 embeddings
Links: 50 deduped, 48 chunks with images


# Compute BM25 sparse vectors for a small test batch

In [5]:
test_chunks = list(who_chunk_lookup.values())[:5]

test_texts = [c.text for c in test_chunks]
test_sparse_vectors = list(model.embed(test_texts))

for chunk, sparse_vec in zip(test_chunks, test_sparse_vectors):
    print(f"chunk_id={chunk.chunk_id}")
    print(f"  text preview: {chunk.text[:80]}...")
    print(f"  non-zero terms: {len(sparse_vec.indices)}")
    print()

chunk_id=tuberculosis_who_text_0
  text preview: WHO consolidated guidelines on tuberculosis: Module 4 Treatment: WHO
consolidate...
  non-zero terms: 61

chunk_id=tuberculosis_who_text_1
  text preview: WHO consolidated guidelines on tuberculosis: Module 4 Treatment: If you adapt th...
  non-zero terms: 101

chunk_id=tuberculosis_who_text_2
  text preview: WHO consolidated guidelines on tuberculosis: Module 4 Treatment: General disclai...
  non-zero terms: 84

chunk_id=tuberculosis_who_text_3
  text preview: WHO consolidated guidelines on tuberculosis: Module 4 Treatment: Design by Inis ...
  non-zero terms: 87

chunk_id=tuberculosis_who_text_4
  text preview: WHO consolidated guidelines on tuberculosis: Module 4 Treatment: 2010 and 2017 D...
  non-zero terms: 71



# Build points with both dense and sparse vectors, test-upload a small batch

In [6]:
from qdrant_client.http import models as qmodels

def build_hybrid_point(chunk, dense_vector, sparse_vector, chunk_to_images_map=None):
    payload = {
        "chunk_id": chunk.chunk_id,
        "source": chunk.source,
        "topics": chunk.topics,
        "source_id": chunk.source_id,
        "chunk_type": chunk.chunk_type,
        "chunk_index": chunk.chunk_index,
        "text": chunk.text,
        "raw_text": chunk.raw_text,
        "metadata": chunk.metadata or {},
        "linked_images": (chunk_to_images_map or {}).get(chunk.chunk_id, []),
    }
    return qmodels.PointStruct(
        id=chunk.point_id,
        vector={
            "dense": dense_vector.tolist(),
            "sparse": qmodels.SparseVector(
                indices=sparse_vector.indices.tolist(),
                values=sparse_vector.values.tolist(),
            ),
        },
        payload=payload,
    )

# small test batch: 10 WHO chunks
test_rows = list(zip(who_index[:10], who_embeddings[:10]))
test_texts = []
test_chunks_for_batch = []
for row, vec in test_rows:
    chunk = who_chunk_lookup.get(row["chunk_id"])
    if chunk:
        test_chunks_for_batch.append((chunk, vec))
        test_texts.append(chunk.text)

test_sparse = list(model.embed(test_texts))

hybrid_test_points = [
    build_hybrid_point(chunk, vec, sparse_vec, chunk_to_images)
    for (chunk, vec), sparse_vec in zip(test_chunks_for_batch, test_sparse)
]

print(f"Built {len(hybrid_test_points)} hybrid test points")
client.upsert(collection_name=TEXT_COLLECTION, points=hybrid_test_points)

Built 10 hybrid test points


UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

# Search using sparse vector only, verify BM25-style keyword matching

In [8]:
query_text = "tuberculosis treatment guidelines"
query_sparse = list(model.embed([query_text]))[0]

sparse_results = client.query_points(
    collection_name=TEXT_COLLECTION,
    query=qmodels.SparseVector(
        indices=query_sparse.indices.tolist(),
        values=query_sparse.values.tolist(),
    ),
    using="sparse",
    limit=5,
).points

for r in sparse_results:
    print(f"score={r.score:.4f}  chunk_id={r.payload['chunk_id']}")

score=0.4682  chunk_id=tuberculosis_who_text_0
score=0.4462  chunk_id=tuberculosis_who_text_4
score=0.4293  chunk_id=tuberculosis_who_text_5
score=0.4129  chunk_id=tuberculosis_who_text_7
score=0.4052  chunk_id=tuberculosis_who_text_8


# Search using dense vector only, confirm both vector types coexist correctly

In [9]:
dense_query_vector = hybrid_test_points[0].vector["dense"]

dense_results = client.query_points(
    collection_name=TEXT_COLLECTION,
    query=dense_query_vector,
    using="dense",
    limit=5,
).points

for r in dense_results:
    print(f"score={r.score:.4f}  chunk_id={r.payload['chunk_id']}")

score=1.0000  chunk_id=tuberculosis_who_text_0
score=0.8565  chunk_id=tuberculosis_who_text_1
score=0.8441  chunk_id=tuberculosis_who_text_2
score=0.8334  chunk_id=tuberculosis_who_text_4
score=0.8060  chunk_id=tuberculosis_who_text_5


# Full-scale hybrid upload, all three sources

In [10]:
def upload_source_hybrid(source_name, chunk_lookup, embeddings_arr, index_rows, chunk_to_images_map=None, batch_size=250):
    rows = [(row, vec) for row, vec in zip(index_rows, embeddings_arr) if row["chunk_id"] in chunk_lookup]
    print(f"{source_name}: {len(rows)} points to upload")

    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        chunks = [chunk_lookup[row["chunk_id"]] for row, _ in batch]
        texts = [c.text for c in chunks]
        sparse_vecs = list(model.embed(texts))

        points = [
            build_hybrid_point(chunk, vec, sparse_vec, chunk_to_images_map)
            for (row, vec), chunk, sparse_vec in zip(batch, chunks, sparse_vecs)
        ]
        client.upsert(collection_name=TEXT_COLLECTION, points=points)
        print(f"  {source_name}: uploaded {min(i + batch_size, len(rows))}/{len(rows)}")

    return len(rows)

total = 0
total += upload_source_hybrid("who", who_chunk_lookup, who_embeddings, who_index, chunk_to_images_map=chunk_to_images)
total += upload_source_hybrid("pubmed", pubmed_chunk_lookup, pubmed_embeddings, pubmed_index)
total += upload_source_hybrid("openfda", openfda_chunk_lookup, openfda_embeddings, openfda_index)

print(f"\nTotal uploaded: {total}")

who: 4804 points to upload
  who: uploaded 250/4804
  who: uploaded 500/4804
  who: uploaded 750/4804
  who: uploaded 1000/4804
  who: uploaded 1250/4804
  who: uploaded 1500/4804
  who: uploaded 1750/4804
  who: uploaded 2000/4804
  who: uploaded 2250/4804
  who: uploaded 2500/4804
  who: uploaded 2750/4804
  who: uploaded 3000/4804
  who: uploaded 3250/4804
  who: uploaded 3500/4804
  who: uploaded 3750/4804
  who: uploaded 4000/4804
  who: uploaded 4250/4804
  who: uploaded 4500/4804
  who: uploaded 4750/4804
  who: uploaded 4804/4804
pubmed: 4725 points to upload
  pubmed: uploaded 250/4725
  pubmed: uploaded 500/4725
  pubmed: uploaded 750/4725
  pubmed: uploaded 1000/4725
  pubmed: uploaded 1250/4725
  pubmed: uploaded 1500/4725
  pubmed: uploaded 1750/4725
  pubmed: uploaded 2000/4725
  pubmed: uploaded 2250/4725
  pubmed: uploaded 2500/4725
  pubmed: uploaded 2750/4725
  pubmed: uploaded 3000/4725
  pubmed: uploaded 3250/4725
  pubmed: uploaded 3500/4725
  pubmed: uploaded 3750

# Full-corpus verification

In [11]:
count = client.count(collection_name=TEXT_COLLECTION, exact=True)
print(f"medrag_text count: {count.count} (expected 22696)")
assert count.count == 22696, "Count mismatch!"
print("Verified.")

medrag_text count: 22696 (expected 22696)
Verified.


In [12]:
query_text = "metformin diabetes dosage"
query_sparse = list(model.embed([query_text]))[0]

results = client.query_points(
    collection_name=TEXT_COLLECTION,
    query=qmodels.SparseVector(
        indices=query_sparse.indices.tolist(),
        values=query_sparse.values.tolist(),
    ),
    using="sparse",
    limit=5,
).points

for r in results:
    print(f"score={r.score:.4f}  source={r.payload['source']}  chunk_id={r.payload['chunk_id']}")

score=28.4833  source=openfda  chunk_id=Synjardy_openfda_26
score=27.2849  source=openfda  chunk_id=Jardiance_openfda_23
score=25.4029  source=openfda  chunk_id=Synjardy_openfda_34
score=25.2941  source=openfda  chunk_id=Metformin Hydrochloride_openfda_4
score=24.9830  source=openfda  chunk_id=ZITUVIMET_openfda_25
